# Impacto financiero y ambiental (ESG) de la gestión de mercancías falsificadas
## De la destrucción tradicional a la economía circular

**Autor/a:** _(tu nombre)_ · **Proyecto final del bootcamp de Data Analytics**

### Premisa
Cuando la aduana retiene mercancía falsificada en la frontera de la UE (Reglamento (UE) 608/2013), el titular de la marca suele asumir el coste de almacenarla y destruirla. Este proyecto evalúa **cuánto cuesta y cuánto CO₂ emite ese proceso, y desde qué volumen reciclar sería mejor opción**.

### Datos
- **Reales (CBP, EE. UU., FY2023-24):** sirven para calibrar la forma de los envíos (unidades, vías de transporte, líneas por envío).
- **Tarifas (EUIPO 2023), plazos legales, factores IPCC y pesos PEFCR:** parámetros documentados.
- **Dataset sintético** de 12.000 envíos generado con esos parámetros (`generador_dataset.py`), con costes calculados para 15 países y dos escenarios: **S1** (destrucción con mínimo de 1 t por envío) y **S2** (proporcional al peso).

> Los datos son una aproximación: ni las propias autoridades conocen el peso real de lo falsificado. Por eso el análisis se centra en **umbrales** y **rangos**, no en cifras exactas.

### Hipótesis
| | Pregunta | Cómo se responde |
|---|---|---|
| **H1** | ¿Desde qué volumen (Q\*) reciclar es más barato y emite menos que destruir? | Regresión log-log, punto de equilibrio con sensibilidad, escenarios de CO₂ |
| **H2** | ¿A partir de cuántos días de custodia (d\*) almacenar cuesta más que destruir? | Punto de equilibrio, comparación por circuito, cohortes |
| **H3** | ¿Cómo cambia el material dominante el umbral ambiental? | Reparto peso/emisiones, factor medio por material, umbral por material |

### Índice
0. Guía rápida de los análisis
1. Limpieza de los datos reales (CBP)
2. Carga y validación del dataset sintético
3. Exploración (EDA)
4. H1: volumen mínimo para reciclar
5. H2: días de custodia frente a coste de destruir
6. H3: material dominante
7. Gráficas finales
8. Conclusiones y límites

## Guía rápida: qué análisis se hacen y por qué

| Análisis | Qué hace | Por qué se usa | Qué se espera |
|---|---|---|---|
| Validación del dataset | Cuenta nulos y duplicados y comprueba que las columnas calculadas cuadren | El dataset es sintético: se valida, no se limpia | Cero errores |
| Regresión OLS log-log (H1) | Mide cuánto sube el coste al subir el peso | Confirma el efecto del mínimo de 1 t | Pendiente ≈ 0 por debajo de 1 t y ≈ 1 por encima |
| Punto de equilibrio Q\* (H1) | Calcula el peso del envío desde el que reciclar es más barato que destruir | Responde directamente a H1 | Q\* sube con el coste fijo F y el variable v |
| Sensibilidad y escenarios de CO₂ (H1) | Repite el cálculo con varios costes y reducciones de CO₂ | No hay fuente pública para F y v: se da un abanico, no un valor | Resultados que dependen mucho del escenario |
| Punto de equilibrio d\* (H2) | Calcula los días de almacenaje que costarían lo mismo que destruir | Responde directamente a H2 | d\* muy por encima de la custodia típica |
| Comparación por circuito y cohortes (H2) | Repite la comparación por circuito y por trimestre | Ver si el resultado cambia según el circuito o el tiempo | Litigioso con más custodia; sin tendencia temporal (fechas simuladas) |
| Correlación peso–d\* (H2) | Mide si los envíos más pesados llegan antes al umbral | Explica por qué d\* difiere entre circuitos | Correlación negativa |
| Reparto y umbral por material (H3) | Compara peso y emisiones por material y calcula las toneladas de envío para evitar 1 tCO₂ | Mostrar que el peso y el impacto ambiental no coinciden | El plástico emite mucho más CO₂ por tonelada |

**No se usan contrastes clásicos** (t de Student, chi², ANOVA): sobre datos sintéticos serían circulares, porque detectarían la estructura que ya puso el generador.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from matplotlib.patches import Patch
from matplotlib.ticker import FixedLocator, FixedFormatter

pd.set_option("display.width", 160)

## 1. Limpieza de los datos reales (CBP)

**Objetivo:** pasar del archivo bruto de incautaciones del CBP (FY2023-24) a una tabla de **envíos de moda** con una fila por envío, que es la base con la que calibramos el dataset sintético.

**Qué se hace en cada paso:**
1. Quedarse con unidades contables (EA, PC, PR): se descartan kilos, litros, metros, etc.
2. Quitar líneas sin cantidad o sin valor (cantidad > 0 y MSRP > 0).
3. Quedarse con las cuatro categorías de moda: ropa y accesorios, bolsos, relojes y joyería, calzado.
4. Regla de valores extremos: se descartan los valores de relleno (MSRP ≤ 1 o menos de 1 $ por unidad) y los valores por encima del percentil 99,9 en calzado y ropa. **Solo afecta a la calibración del valor por unidad**, no a qué envíos se conservan.
5. Agregar por envío, asignando la categoría dominante por unidades.

> Esta celda necesita el archivo `ipr-seizures-fy23-fy24.csv` del CBP. Si no lo tienes cargado, se salta y el resto del notebook funciona igual (usa solo el dataset sintético).

In [ ]:
RUTA_CBP = "ipr-seizures-fy23-fy24.csv"
moda = ["Wearing Apparel/Accessories", "Handbags/Wallets", "Watches/Jewelry", "Footwear"]

try:
    cbp = pd.read_csv(RUTA_CBP)
    pasos = []

    def anota(nombre, d):
        pasos.append({"paso": nombre, "lineas": len(d),
                      "envios_unicos": d["Unique Seizure ID"].nunique()})

    anota("0. Datos en bruto (CBP FY2023-24)", cbp)

    u = cbp[cbp["Unit of Measure"].isin(["EA", "PC", "PR"])]
    anota("1. Unidades contables (EA, PC, PR)", u)

    v = u[(u["Quantity Seized"] > 0) & (u["MSRP"] > 0)].copy()
    anota("2. Cantidad y valor > 0", v)

    lm = v[v["Product"].isin(moda)].copy()
    anota("3. Líneas de moda (4 categorías)", lm)

    # 4. Regla de valores extremos (solo afecta al valor por unidad)
    lm["vu"] = lm["MSRP"] / lm["Quantity Seized"]
    lm["hi"] = lm.groupby("Product")["vu"].transform(lambda s: s.quantile(.999))
    relleno = (lm["MSRP"] <= 1) | (lm["vu"] < 1)
    alto = (lm["vu"] > lm["hi"]) & lm["Product"].isin(["Footwear", "Wearing Apparel/Accessories"])
    limpio = lm[~(relleno | alto)]
    anota("4. Sin valores extremos (calibración del valor)", limpio)

    # 5. Envíos de moda (categoría dominante por unidades)
    dom = (v.groupby(["Unique Seizure ID", "Product"])["Quantity Seized"].sum()
             .reset_index().sort_values("Quantity Seized", ascending=False)
             .drop_duplicates("Unique Seizure ID").rename(columns={"Product": "categoria"})
             [["Unique Seizure ID", "categoria"]])
    inc = (v.groupby("Unique Seizure ID")
             .agg(fy=("FY", "first"), transporte=("Mode of Transportation", "first"),
                  unidades=("Quantity Seized", "sum"), valor=("MSRP", "sum"),
                  n_lineas=("Product", "size")).reset_index().merge(dom, on="Unique Seizure ID"))
    inc_moda = inc[inc["categoria"].isin(moda)]
    pasos.append({"paso": "5. Envíos de moda agregados",
                  "lineas": int(inc_moda["n_lineas"].sum()), "envios_unicos": len(inc_moda)})

    tabla = pd.DataFrame(pasos)
    tabla["% lineas sobre bruto"] = (tabla["lineas"] / tabla.loc[0, "lineas"] * 100).round(1)
    print(tabla.to_string(index=False))
    print("Líneas descartadas por la regla de extremos:", int((relleno | alto).sum()),
          "de", len(lm), f"({(relleno | alto).mean()*100:.1f} %)")
except FileNotFoundError:
    print("No se encontró", RUTA_CBP, "→ se salta la sección 1 (los resultados están en el texto de abajo).")

**Lectura (valores obtenidos con el archivo del CBP):**

| Paso | Líneas | Envíos únicos | % líneas sobre bruto |
|---|---|---|---|
| 0. Datos en bruto | 188.518 | 40.353 | 100,0 |
| 1. Unidades contables | 188.097 | 40.252 | 99,8 |
| 2. Cantidad y valor > 0 | 188.048 | 40.235 | 99,8 |
| 3. Líneas de moda | 153.324 | 28.156 | 81,3 |
| 4. Sin valores extremos | 147.459 | 27.938 | 78,2 |
| 5. Envíos de moda agregados | 152.788 | 27.340 | 81,0 |

- El filtro de **categoría de moda** es el que más elimina (≈34.700 líneas, el 18,5 % del bruto). Los dos primeros apenas tocan nada (0,2 %).
- La regla de extremos descarta el **3,8 %** de las líneas de moda (5.865 de 153.324), y solo afecta a la calibración del valor por unidad.
- Resultado: **27.340 envíos de moda** (67,8 % de los 40.353 iniciales), una fila por envío.

## 2. Carga y validación del dataset sintético

El dataset sintético está **limpio por construcción** (lo genera un script con reglas explícitas), así que aquí no se "limpia": se **valida**. Comprobamos tipos, nulos, duplicados y que las columnas calculadas sean coherentes entre sí.

Archivos:
- `envios_sinteticos.csv`: 12.000 envíos × 27 columnas.
- `costes_por_pais.csv`: 15 países × 12.000 envíos = 180.000 filas, con costes S1 y S2.

In [ ]:
envios = pd.read_csv("envios_sinteticos.csv", parse_dates=["fecha_decomiso", "fecha_destruccion"])
costes = pd.read_csv("costes_por_pais.csv")
print(envios.shape, costes.shape)
envios.info()

In [ ]:
# Comprobaciones de calidad
print("Filas duplicadas:", envios.duplicated().sum())
print("IDs repetidos:", envios["id_envio"].duplicated().sum())
print("Custodia = trámite + procedimiento + logística:",
      (envios["dias_tramite"] + envios["dias_procedimiento"] + envios["dias_logistica"]
       == envios["dias_custodia"]).all())
print("Fechas coherentes con los días:",
      ((envios["fecha_destruccion"] - envios["fecha_decomiso"]).dt.days
       == envios["dias_custodia"]).all())

# Distribuciones
num = ["unidades", "peso_kg", "n_palets", "dias_custodia"]
print(envios[num].describe(percentiles=[.5, .9, .99]).round(2))

for c in ["via_transporte", "categoria", "circuito", "pais_origen"]:
    print((envios[c].value_counts(normalize=True) * 100).round(1), "\n")

**Lectura:**
- Sin nulos, sin duplicados, tipos correctos, custodia y fechas coherentes.
- Distribución muy asimétrica: mediana de 6 unidades y 2,1 kg, pero media de 203 unidades y 86 kg. La mayoría de envíos es diminuta y unos pocos son enormes (máximo ≈105 t).
- Reparto por circuito: 63,7 % simplificado, 28,0 % estándar, 8,4 % litigioso (ancladas al informe de la Comisión).
- Origen: China 40,8 %, otros 33,7 %, Turquía 19,5 %, Hong Kong 5,9 %.

## 3. Exploración (EDA): concentración del peso

Antes de contrastar nada, conviene saber **cómo se reparte el peso entre envíos**, porque el coste de destruir depende del peso.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 3.6))
axs[0].hist(envios["peso_kg"], bins=60, color="#1f4e79")
axs[0].set_title("Peso por envío (escala lineal)"); axs[0].set_xlabel("kg")
axs[1].hist(np.log10(envios["peso_kg"]), bins=60, color="#1f4e79")
axs[1].set_title("Peso por envío (log10)"); axs[1].set_xlabel("log10(kg)")
plt.tight_layout(); plt.show()

p = np.sort(envios["peso_kg"].values)[::-1]
top1 = p[: int(len(p) * 0.01)].sum() / p.sum() * 100
print(f"El 1 % de envíos más pesados = {top1:.1f} % del peso total")
print("Envíos de más de 1 t:", (envios["peso_kg"] > 1000).sum(),
      "| peso total (t):", round(envios["peso_kg"].sum() / 1000))

**Lectura:** el **1 % de los envíos concentra el 69 % del peso** total (≈1.030 t) y solo 133 envíos (1,1 %) superan 1 t. Con un coste de destrucción que depende del peso, esto anticipa que unos pocos envíos grandes dominan tanto el coste como las emisiones.

## 4. H1: ¿desde qué volumen compensa reciclar?

**Pregunta:** ¿existe un peso mínimo de envío (Q\*) a partir del cual reciclar es más barato y emite menos que destruir?

**Delimitación:**
- **Financiero:** coste que asume el titular desde la retención hasta la destrucción o el reciclaje (art. 29). El coste del sistema se reporta aparte.
- **Ambiental:** emisiones directas de la incineración (IPCC 2006: textil 0,293; caucho y cuero 0,413; plásticos 2,75 tCO₂/t). Sin créditos por material recuperado.
- **Referencia:** España (destrucción 400 €/t en el modelo). Escenario S1.

### 4.1 Regresión log-log: ¿cuánto sube el coste cuando sube el peso?
Excluimos el circuito simplificado, donde el coste para el titular es 0.

In [ ]:
es = costes[costes["iso2"] == "ES"]
df = envios.merge(es, on="id_envio")
df = df[df["coste_titular_s1_eur"] > 0].copy()
df["coste_por_kg"] = df["coste_titular_s1_eur"] / df["peso_kg"]

plt.figure(figsize=(7, 4))
plt.scatter(df["peso_kg"], df["coste_por_kg"], s=4, alpha=.3, color="#1f4e79")
plt.xscale("log"); plt.yscale("log")
plt.xlabel("Peso del envío (kg)"); plt.ylabel("Coste por kg (€)")
plt.title("Coste por kg frente a peso (España, S1)")
plt.tight_layout(); plt.show()

In [ ]:
for nombre, sub in [("Menos de 1 t", df[df["peso_kg"] < 1000]),
                    ("1 t o más", df[df["peso_kg"] >= 1000])]:
    m = smf.ols("np.log10(coste_titular_s1_eur) ~ np.log10(peso_kg)", data=sub).fit()
    print(f"{nombre}: n={len(sub):,} | pendiente={m.params.iloc[1]:.2f} | R²={m.rsquared:.2f}")

**Lectura:**

| Tramo | Envíos | Pendiente | R² |
|---|---|---|---|
| Menos de 1 t | 8.512 | 0,01 | 0,08 |
| 1 t o más | 133 | 0,98 | 0,93 |

Por debajo de 1 t el coste **no depende del peso** (se paga el mínimo de 1 t); por encima es casi **proporcional**. Esto describe el modelo de costes (tarifa × peso con mínimo), no un comportamiento observado del mundo real.

### 4.2 Punto de equilibrio Q\*
**Notación.** **Q** es la *cantidad*: el peso del envío en toneladas (convención económica). **Q\*** es el peso crítico (el asterisco marca el valor de equilibrio): el peso del envío a partir del cual reciclar cuesta menos que destruir. **F** es el coste fijo de reciclar un envío (€), **v** el coste variable por tonelada reciclada (€/t), **f** la fracción reciclable del envío y **t** la tarifa de destrucción (€/t).

Reciclar un envío de peso Q cuesta **F + v × (f × Q)**: el coste fijo más el variable, que solo se paga por la parte reciclable. A cambio se evita destruir esa parte, lo que ahorra **t × f × Q**. Solo una fracción **f = 34 %** del envío es recuperable (referencia: PVH Re-Calvin). Reciclar compensa cuando el ahorro supera el coste, es decir, cuando **t × f × Q > F + v × f × Q**. Despejando Q:

**Q\* = F / (f × (t − v))**

Como F y v no tienen fuente pública, se presenta una rejilla de sensibilidad: F ∈ {500; 2.000; 5.000} € por envío y v ∈ {0; 150; 300} €/t, con t = 400 €/t.

In [ ]:
f, t = 0.34, 400
Fs = [500, 2000, 5000]
vs = [0, 150, 300]

def qstar(F, v, t=t, f=f):
    return F / (f * (t - v))

filas = []
for F in Fs:
    for v in vs:
        q = qstar(F, v)
        sobre = envios[envios["peso_kg"] / 1000 > q]
        filas.append({"F (€/envío)": F, "v (€/t)": v, "Q* (t)": round(q, 1),
                      "envíos sobre Q*": len(sobre),
                      "% del peso": round(sobre["peso_kg"].sum() / envios["peso_kg"].sum() * 100, 1)})
print(pd.DataFrame(filas).to_string(index=False))

**Lectura:** Q\* va de **3,7 t** (F = 500, v = 0) a **147,1 t** (F = 5.000, v = 300). Superan el umbral 35 envíos en el caso favorable (52 % del peso), 7 en el intermedio (28 %) y ninguno en el desfavorable. El umbral existe, pero depende mucho del coste real del reciclaje, que no conocemos.

### 4.3 CO₂ evitado por escenario
Emisiones de los envíos que superan Q\*, multiplicadas por la fracción reciclable (34 %) y por la reducción de CO₂ supuesta (10, 30 y 60 %).

In [ ]:
umbrales = {"favorable": qstar(500, 0), "intermedio": qstar(2000, 150), "desfavorable": qstar(5000, 300)}
total_emis = envios["emisiones_incineracion_tco2"].sum()
print(f"Emisiones totales: {total_emis:,.0f} tCO2\n")

filas = []
for esc, q in umbrales.items():
    sobre = envios[envios["peso_kg"] / 1000 > q]
    for red in (0.10, 0.30, 0.60):
        evitado = sobre["emisiones_incineracion_tco2"].sum() * f * red
        filas.append({"escenario": esc, "Q* (t)": round(q, 1), "reducción CO2e": f"{int(red*100)} %",
                      "tCO2 evitadas": round(evitado, 2), "% del total": round(evitado / total_emis * 100, 2)})
print(pd.DataFrame(filas).to_string(index=False))

**Lectura:** incluso en el mejor caso solo se evita entre el **2 % y el 12 %** (2,1 / 6,2 / 12,4 %) de las emisiones totales, porque solo el 34 % del envío es reciclable y solo el 0,3 % de los envíos supera el umbral. Esos envíos concentran, aun así, el 52 % del peso y ≈61 % de las emisiones. (Se usa el umbral exacto, 3,676 t; con el redondeado de 3,7 t saldrían 34 envíos y cifras ligeramente menores.)

**Conclusión provisional de H1:** el umbral existe y depende mucho de F y v. Reciclar compensa en los envíos grandes; para los pequeños haría falta **agruparlos** (módulo previsto para el portfolio).

## 5. H2: días de custodia frente a coste de destruir

**Pregunta:** ¿a partir de cuántos días de custodia el almacenaje cuesta más que destruir?

Para cada envío: **d\* = coste de destrucción / coste diario de almacenaje**. Si los días reales de custodia superan d\*, almacenar sale más caro que destruir.

**Circuitos:** simplificado (el titular acepta la destrucción, coste 0 para él), estándar (sin oposición) y litigioso (el **declarante o consignatario** se opone y el caso va a vía judicial).

In [ ]:
df["alm_dia"] = df["almacenaje_s1_eur"] / df["dias_custodia"]
df["d_star"] = df["destruccion_s1_eur"] / df["alm_dia"]
df["supera"] = df["dias_custodia"] > df["d_star"]

print(df.groupby("circuito")[["dias_custodia", "d_star"]].median())
print(df.groupby("circuito")["supera"].mean().round(3))

**Lectura:** en el estándar la custodia mediana es de 32 días frente a un d\* de ≈533; en el litigioso, 228 frente a ≈1.244. Solo supera d\* el **0,7 %** de los estándar y el **3,4 %** de los litigiosos.

### 5.1 ¿Qué determina d\*? ¿Y cambia con el tiempo?

In [ ]:
print("Correlación log-log peso vs d*:",
      np.corrcoef(np.log10(df["peso_kg"]), np.log10(df["d_star"]))[0, 1].round(2))
print(df.groupby("circuito")["peso_kg"].median())

coh = df.groupby(["trimestre", "circuito"]).agg(
    envios=("id_envio", "count"),
    custodia_mediana=("dias_custodia", "median"),
    pct_supera=("supera", "mean"),
).round(3)
print(coh)

**Lectura:**
- Correlación de **−0,58** entre peso y d\*: los envíos pesados alcanzan antes el umbral. Los litigiosos pesan menos (mediana de 1,9 kg frente a 5,8 kg), y con el mínimo de 1 t destruir les cuesta lo mismo mientras que almacenar cuesta casi nada. Es una consecuencia de las tarifas del modelo.
- Las **cohortes por trimestre** salen planas (estándar 0,3-1,0 %; litigioso 1,5-4,9 %, ruido de muestra con ≈120 envíos por celda). Es lo esperable: las fechas son simuladas y no hay estacionalidad real que detectar.

### 5.2 Peso del almacenaje en el coste del titular

In [ ]:
print("Almacenaje sobre coste total del titular:",
      round(df["almacenaje_s1_eur"].sum() / df["coste_titular_s1_eur"].sum() * 100, 1), "%")

por_circ = df.groupby("circuito").agg(
    envios=("id_envio", "count"),
    alm=("almacenaje_s1_eur", "sum"),
    total=("coste_titular_s1_eur", "sum"),
)
por_circ["pct_envios"] = (por_circ["envios"] / por_circ["envios"].sum() * 100).round(1)
por_circ["pct_del_almacenaje"] = (por_circ["alm"] / por_circ["alm"].sum() * 100).round(1)
por_circ["alm_sobre_total_%"] = (por_circ["alm"] / por_circ["total"] * 100).round(1)
por_circ["alm_por_envio_€"] = (por_circ["alm"] / por_circ["envios"]).round(0)
por_circ["coste_por_envio_€"] = (por_circ["total"] / por_circ["envios"]).round(0)
print(por_circ.round(1))

**Lectura:**
- El almacenaje es solo el **15 %** del coste del titular; el 85 % es destruir.
- El circuito litigioso es el 11,6 % de los envíos con coste y genera el 24,7 % del almacenaje (≈160 € por envío frente a 64 €, ×2,5). El 75 % del almacenaje sigue viniendo del estándar por volumen.
- **Mensaje:** la oposición del declarante multiplica por 2,5 el almacenaje por envío, pero **el grueso del coste está en destruir, no en esperar**.
- **Cautelas:** la duración de los litigios es un supuesto sin fuente; el coste de destrucción está dominado por el mínimo de 1 t (en S2, proporcional al peso, el coste total del titular es 3,8 veces menor).

**Conclusión de H2:** con estas tarifas almacenar supera a destruir en menos del 1 % de los envíos estándar y ≈3 % de los litigiosos, sin efecto temporal.

## 6. H3: el material dominante

**Pregunta:** ¿cómo cambia el material dominante el umbral **ambiental** de H1?

El umbral financiero es el mismo para todos los materiales (solo existe tarifa de textil). Lo que cambia es el CO₂ que se evita al reciclar una tonelada.

> El material de cada envío se **asigna a partir de su categoría** (supuesto declarado). Los resultados sobre material son consecuencia del modelo; lo robusto es la física: el plástico emite ≈10 veces más CO₂ por tonelada que el textil al incinerarse.

In [ ]:
mat = envios.groupby("material_dominante").agg(
    envios=("id_envio", "count"),
    peso_t=("peso_kg", lambda x: x.sum() / 1000),
    emis_tco2=("emisiones_incineracion_tco2", "sum"),
)
mat["pct_peso"] = (mat["peso_t"] / mat["peso_t"].sum() * 100).round(1)
mat["pct_emis"] = (mat["emis_tco2"] / mat["emis_tco2"].sum() * 100).round(1)
mat["ef_medio_tco2_t"] = (mat["emis_tco2"] / mat["peso_t"]).round(2)
print(mat.round(2))

**Lectura:** el plástico sintético pesa el 33 % y emite el **81 %** del CO₂. El textil pesa más (38 %) pero emite el 10 %. El factor medio real (emisiones / peso) es ≈2,79 (plástico), 0,39 (caucho y cuero) y 0,30 (textil) tCO₂/t, coherente con el IPCC.

### 6.1 ¿Dónde estaría el beneficio de reciclar, por material?
CO₂ evitado (escenario de reducción del 30 %) solo en los envíos que superan el umbral favorable.

In [ ]:
red = 0.30
grandes = envios[envios["peso_kg"] / 1000 > umbrales["favorable"]]
res = grandes.groupby("material_dominante").agg(
    envios=("id_envio", "count"),
    peso_t=("peso_kg", lambda x: x.sum() / 1000),
    emis=("emisiones_incineracion_tco2", "sum"),
)
res["evitado_30pct"] = (res["emis"] * f * red).round(1)
res["pct_del_evitado"] = (res["evitado_30pct"] / res["evitado_30pct"].sum() * 100).round(1)
print(res.round(1))

**Lectura:** de los 35 envíos por encima del umbral favorable (3,676 t), **10 de plástico aportan el 87,0 %** del CO₂ evitado; los 9 de caucho y cuero, el 6,6 %, y los 16 de textil, el 6,4 %. Lo que decide el beneficio es el material, no el número de envíos.

### 6.2 Umbral ambiental por material
Toneladas que habría que reciclar para evitar **1 tCO₂** (fracción reciclable 34 %, reducción 30 %).

In [ ]:
ef_medio = mat["emis_tco2"] / mat["peso_t"]
evitado_por_t = (ef_medio * f * red).round(3)
lote_1tco2 = (1 / evitado_por_t.replace(0, float("nan"))).round(1)
print(pd.DataFrame({"ef_medio": ef_medio.round(2),
                    "tCO2_evitadas_por_t": evitado_por_t,
                    "t_para_evitar_1tCO2": lote_1tco2}))

**Lectura:** hacen falta ≈**3,5 t** de plástico, ≈25,6 t de caucho y cuero y ≈32,3 t de textil para evitar 1 tCO₂. **El umbral financiero es único, el ambiental cambia unas diez veces según el material.** El metal no emite en la incineración (no hay dato).

**Conclusión de H3:** el material dominante desplaza el umbral ambiental de H1; el beneficio de reciclar se concentra en los envíos grandes de plástico.

## 7. Gráficas finales

Cuatro gráficas para el informe, una por mensaje clave. Cada una se guarda como PNG (200 dpi).

### Gráfica 1. El 1 % de los envíos concentra el 69 % del peso

In [ ]:
p = np.sort(envios["peso_kg"].values)[::-1]
acum = p.cumsum() / p.sum() * 100
x = np.arange(1, len(p) + 1) / len(p) * 100

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(x, acum, color="#1f4e79", lw=2)
ax.scatter([1], [69.1], color="#c0392b", zorder=3)
ax.annotate("1 % de los envíos\n= 69 % del peso", xy=(1, 69.1),
            xytext=(3, 45), arrowprops=dict(arrowstyle="->", color="grey"))
ax.set_xscale("log")
ax.xaxis.set_major_locator(FixedLocator([0.01, 0.1, 1, 10, 100]))
ax.xaxis.set_major_formatter(FixedFormatter(["0,01 %", "0,1 %", "1 %", "10 %", "100 %"]))
ax.minorticks_off()
ax.set_xlabel("Envíos ordenados del más pesado al más ligero (escala logarítmica)")
ax.set_ylabel("% acumulado del peso total")
ax.set_title("El 1 % de los envíos concentra el 69 % del peso")
ax.set_xlim(0.01, 100); ax.set_ylim(0, 100)
plt.tight_layout()
plt.savefig("g1_concentracion_peso.png", dpi=200)
plt.show()

### Gráfica 2. Lote mínimo (Q\*) y envíos que lo superan

In [ ]:
Q = np.array([[qstar(F, v) for F in Fs] for v in vs])
n_sobre = np.array([[(envios["peso_kg"] / 1000 > q).sum() for q in fila] for fila in Q])

fig, ax = plt.subplots(figsize=(7, 4))
ax.imshow(Q, cmap="RdYlGn_r", aspect="auto")
for i in range(3):
    for j in range(3):
        txt = f"{Q[i, j]:.1f} t\n{n_sobre[i, j]} envíos".replace(".", ",")
        color = "white" if (Q[i, j] < 15 or Q[i, j] > 100) else "black"
        ax.text(j, i, txt, ha="center", va="center", fontsize=10, color=color)
ax.set_xticks(range(3)); ax.set_xticklabels([f"{F:,} €".replace(",", ".") for F in Fs])
ax.set_yticks(range(3)); ax.set_yticklabels([f"{v} €/t" for v in vs])
ax.set_xlabel("Coste fijo por envío (F)")
ax.set_ylabel("Coste variable (v)")
ax.set_title("Lote mínimo (Q*) para que reciclar compense, y envíos que lo superan")
plt.tight_layout()
plt.savefig("g2_mapa_calor_qstar.png", dpi=200)
plt.show()

### Gráfica 3. Almacenar casi nunca supera al coste de destruir
**Cómo leerla:** la barra azul son los días que el envío pasa realmente en custodia; la roja, los días de almacenaje que costarían lo mismo que destruir. Mientras la azul esté por debajo de la roja, almacenar es más barato que destruir.

In [ ]:
med = df.groupby("circuito")[["dias_custodia", "d_star"]].median().loc[["estandar", "litigioso"]]
x = np.arange(2)
fig, ax = plt.subplots(figsize=(7, 4))
b1 = ax.bar(x - 0.2, med["dias_custodia"], 0.4, color="#1f4e79", label="Días reales en custodia (mediana)")
b2 = ax.bar(x + 0.2, med["d_star"], 0.4, color="#c0392b",
            label="Días de almacenaje que costarían lo mismo que destruir (mediana)")
for barras in (b1, b2):
    for b in barras:
        ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 20,
                f"{b.get_height():,.0f}".replace(",", "."), ha="center")
ax.set_xticks(x); ax.set_xticklabels(["Estándar", "Litigioso"])
ax.set_ylabel("Días")
ax.set_ylim(0, med["d_star"].max() * 1.25)
ax.set_title("Almacenar casi nunca supera al coste de destruir")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.savefig("g3_custodia_vs_dstar_barras.png", dpi=200)
plt.show()

**Versión para el anexo (diagrama de cajas):** muestra la dispersión, no solo la mediana. Valores atípicos no representados; sin circuito simplificado (coste para el titular = 0).

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
orden = ["estandar", "litigioso"]
datos_c = [df.loc[df["circuito"] == c, "dias_custodia"] for c in orden]
datos_d = [df.loc[df["circuito"] == c, "d_star"] for c in orden]
pos = np.arange(len(orden))
ax.boxplot(datos_c, positions=pos - 0.18, widths=0.3, patch_artist=True,
           boxprops=dict(facecolor="#1f4e79"), medianprops=dict(color="white"), showfliers=False)
ax.boxplot(datos_d, positions=pos + 0.18, widths=0.3, patch_artist=True,
           boxprops=dict(facecolor="#c0392b"), medianprops=dict(color="white"), showfliers=False)
ax.set_yscale("log")
ax.set_xticks(pos); ax.set_xticklabels(["Estándar", "Litigioso"])
ax.legend(handles=[Patch(color="#1f4e79", label="Días de custodia"),
                   Patch(color="#c0392b", label="Umbral d*")], loc="upper left")
ax.yaxis.set_major_locator(FixedLocator([10, 100, 1000]))
ax.yaxis.set_major_formatter(FixedFormatter(["10", "100", "1.000"]))
ax.minorticks_off()
ax.set_ylabel("Días (escala logarítmica)")
ax.set_title("Almacenar casi nunca supera al coste de destruir")
plt.tight_layout()
plt.savefig("g3b_custodia_vs_dstar_cajas.png", dpi=200)
plt.show()

### Gráfica 4. El plástico pesa un tercio, pero emite más de ocho décimas del CO₂

In [ ]:
m4 = envios.groupby("material_dominante").agg(
    peso=("peso_kg", "sum"), emis=("emisiones_incineracion_tco2", "sum"))
m4["pct_peso"] = m4["peso"] / m4["peso"].sum() * 100
m4["pct_emis"] = m4["emis"] / m4["emis"].sum() * 100
orden = ["textil", "plastico_sintetico", "caucho_cuero", "metal_otros"]
nombres = ["Textil", "Plástico sintético", "Caucho y cuero", "Metal y otros"]
m4 = m4.loc[orden]

x = np.arange(len(orden))
fig, ax = plt.subplots(figsize=(7, 4))
b1 = ax.bar(x - 0.2, m4["pct_peso"], 0.4, color="#7f8c8d", label="% del peso")
b2 = ax.bar(x + 0.2, m4["pct_emis"], 0.4, color="#c0392b", label="% de las emisiones de CO₂")
for barras in (b1, b2):
    for b in barras:
        ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 1,
                f"{b.get_height():.1f} %".replace(".", ","), ha="center")
ax.set_xticks(x); ax.set_xticklabels(nombres)
ax.set_ylabel("% del total"); ax.set_ylim(0, 95)
ax.set_title("El plástico pesa un tercio, pero emite más de ocho décimas del CO₂")
ax.legend(loc="upper right")
plt.tight_layout()
plt.savefig("g4_peso_vs_emisiones_material.png", dpi=200)
plt.show()

## 8. Conclusiones y límites

### Conclusiones por hipótesis
| | Resultado |
|---|---|
| **H1** | Existe un peso mínimo de envío Q\* (de 3,7 a 147 t según F y v). En el mejor caso solo el 0,3 % de los envíos lo supera, pero concentran el 52 % del peso y ≈61 % de las emisiones. El CO₂ evitado va del 0 al 12 % del total. |
| **H2** | Almacenar supera a destruir en menos del 1 % de los envíos estándar y ≈3 % de los litigiosos. El almacenaje es solo el 15 % del coste del titular; la oposición del declarante multiplica por 2,5 el almacenaje por envío. |
| **H3** | El umbral financiero es único; el ambiental cambia ≈10 veces según el material: 3,5 t de plástico evitan 1 tCO₂ frente a 32 t de textil. |

### Límites que hay que declarar
- **Datos sintéticos:** los resultados describen el modelo (tarifas, mínimo de 1 t, supuestos), no un comportamiento observado. Por eso no se usan contrastes clásicos (t, chi², ANOVA): serían circulares.
- **F y v** (coste del reciclaje) no tienen fuente pública: se presenta un abanico, no un valor.
- **Duración de los litigios** y **material por categoría**: supuestos declarados.
- **Límite contable:** se mide el coste del titular y las emisiones directas de la incineración, sin créditos de reciclaje ni ciclo de vida completo.
- Pendiente de confirmar con el mentor: si H1 debe usar el coste del titular o el del sistema.

### Próximos pasos
Power BI (resumen y simulador país × escenario), informe de 4 páginas y presentación. Para el portfolio: Monte Carlo sobre F y v, agrupación de lotes (S2), mapa de calor de 15 países y K-Means.